In [1]:
import pandas as pd
import numpy as np


from sklearn.model_selection   import train_test_split
from sklearn.preprocessing    import LabelEncoder
from sklearn.metrics          import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, hamming_loss
from sklearn.multiclass import OneVsRestClassifier
from tensorflow import keras
from lightgbm import LGBMClassifier
from sklearn.preprocessing    import StandardScaler
from xgboost import XGBClassifier

1.) LOAD DATA INTO A SINGLE DATAFRAME AND INSPECT

In [5]:
#pip install pyarrow

In [4]:
# load product properties in a dataframe
product_props = pd.read_parquet('product_properties.parquet')[['sku','category','price']]

# Function to load the remaining files
def load_event(path, event_name):
    df = pd.read_parquet(path)
    df['event_type'] = event_name
    return df

# Load each event file
buys = load_event('product_buy.parquet', 'buy')
adds = load_event('add_to_cart.parquet', 'add_to_cart')
removes = load_event('remove_from_cart.parquet','remove_from_cart')
visits = load_event('page_visit.parquet', 'page_visit')
searches = load_event('search_query.parquet', 'search_query')



In [8]:
# Create a dtaframe with the events
all_events = pd.concat([buys, adds, removes, visits, searches],
                       axis=0, ignore_index=True)
all_events['timestamp'] = pd.to_datetime(all_events['timestamp'])

In [10]:
# Merge with category & price (product properties)
all_events = all_events.merge(
    product_props,
    on='sku',
    how='left'
)


In [7]:
#all_events

,client_id,timestamp,sku,event_type,url,query,category,price
0,17649961,2022-07-23 20:15:25,18485.0,buy,NaN,NaN,5492.0,72.0
1,16696114,2022-07-11 16:31:30,81192.0,buy,NaN,NaN,6519.0,99.0
2,10238779,2022-05-29 19:35:40,510014.0,buy,NaN,NaN,6350.0,58.0
3,10238779,2022-05-29 19:38:05,510014.0,buy,NaN,NaN,6350.0,58.0
4,10238779,2022-05-29 19:38:05,510014.0,buy,NaN,NaN,6350.0,58.0
...,...,...,...,...,...,...,...,...
168900508,13493699,2022-09-15 13:56:25,NaN,search_query,NaN,[ 82 46 164 200 200 130 24 70 142 94 128 ...,NaN,NaN
168900509,13493699,2022-09-15 13:57:00,NaN,search_query,NaN,[ 82 46 164 200 200 130 24 70 142 94 128 ...,NaN,NaN
168900510,13493699,2022-09-16 01:07:25,NaN,search_query,NaN,[ 82 46 164 200 200 130 24 70 142 94 128 ...,NaN,NaN
168900511,13493699,2022-09-16 01:08:35,NaN,search_query,NaN,[ 82 46 164 200 200 130 24 70 142 94 128 ...,NaN,NaN


In [12]:

# Find the end of data and set cutoff 14 days earlier
data_end  = all_events['timestamp'].max()            
cutoff_dt = data_end - pd.Timedelta(days=14)          

#Display end date and cutoff date
print("Global data end:", data_end)
print("Cutoff date   :", cutoff_dt)



Global data end: 2022-10-10 23:59:59
Cutoff date   : 2022-09-26 23:59:59


In [14]:
# Create a pre cutoff dataframe 
feat_df  = all_events[ all_events['timestamp'] <=  cutoff_dt ].copy()

In [15]:
#Create a post cutoff dataframe
label_df = all_events[(all_events['timestamp'] >   cutoff_dt) &
                     (all_events['timestamp'] <=  data_end)].copy()

print("Feature window: ", feat_df['timestamp'].min(), "→", feat_df['timestamp'].max())
print("Label window:   ", label_df['timestamp'].min(), "→", label_df['timestamp'].max())

Feature window:  2022-05-23 00:10:00 → 2022-09-26 23:59:59
Label window:    2022-09-27 00:00:00 → 2022-10-10 23:59:59


# FIND USERS WHO HAVE BOUGHT IN BOTH PRE AND POST CUTOFF WINDOWS AND MARK THEM AS THE ELIGIBLE USERS AND DROP THE REST

In [18]:
# pre cutoff users
feat_buyers = feat_df.loc[feat_df['event_type']=='buy', 'client_id'].unique()

# post cutoff users
label_buyers = label_df.loc[label_df['event_type']=='buy', 'client_id'].unique()

# Eligible users (intersection of the two)
eligible_users = np.intersect1d(feat_buyers, label_buyers)
print("Total eligible users:", len(eligible_users))

# Drop any users that are not eligible from both dataframes
feat_df  = feat_df[feat_df['client_id'].isin(eligible_users)].copy()
label_df = label_df[label_df['client_id'].isin(eligible_users)].copy()

Total eligible users: 31296


Find users with at least one buy in the post cutoff window

In [20]:
# Restrict to buys in the label window and sort 
label_buys = label_df[label_df['event_type']=='buy'].copy()
label_buys = label_buys[label_buys['client_id'].isin(eligible_users)]
label_buys = label_buys.sort_values(['client_id','timestamp'])

# Get all post cutoff purchases per user 
label_categories = (
    label_buys
      .groupby('client_id')['category']
      .apply(list)
      .reset_index()
      .rename(columns={'category': 'label_categories'})
)

#This should match the eligible users
print("Users with at least one label-window buy:", label_buys['client_id'].nunique())


exploded = label_categories.explode('label_categories')

Users with at least one label-window buy: 31296


FIND THE TOP 15 CATEGORIES IN TERMS OF MOST FREQUENT ONES 

In [76]:
# Find the top 15 categories by frequency 
topK = 15
top_categories = (
    exploded['label_categories']
      .value_counts()
      .head(topK)
      .index
      .tolist()
)
print("Top 15 categories:", top_categories)

# Find the eligible users that have bought from those 15 categories both in pre and post cutoff windows
label_categories = exploded[exploded['label_categories'].isin(top_categories)].copy()
print("After restricting, users left in the label window:", label_categories['client_id'].nunique())

Top 15 categories: [3944.0, 5954.0, 3068.0, 2739.0, 3147.0, 3031.0, 1097.0, 4142.0, 3709.0, 4715.0, 5594.0, 5476.0, 5323.0, 6857.0, 1586.0]
After restricting, users left in the label window: 5610


In [78]:
# Check to see if the users from post cutoff windows match the number of users in pre cutoff window

label_users = label_categories['client_id'].unique()
feat_df = feat_df.loc[feat_df['client_id'].isin(label_users)].copy()
print('users that exist both in feature and label windows: ', feat_df['client_id'].nunique())


users that exist both in feature and label windows:  5610


Create features that aggregate the events per user

In [81]:
event_counts = feat_df.pivot_table(
    index='client_id',
    columns='event_type',
    values='sku',
    aggfunc='count',
    fill_value=0
).add_prefix('count_')

# Unique categories per user 
unique_cats = feat_df.groupby('client_id')['category'] \
                     .nunique() \
                     .rename('unique_categories')

In [83]:
# Generate some statistics (min,max, average) for the categories purchased 

price_stats = (
    feat_df[feat_df['event_type']=='buy']
      .groupby('client_id')['price']
      .agg(avg_price='mean', max_price='max', min_price='min')
)


# Create features regading the activity of each user
last_ts = feat_df.groupby('client_id')['timestamp'].max().rename('last_ts')
first_ts= feat_df.groupby('client_id')['timestamp'].min().rename('first_ts')

recency_days  = ((cutoff_dt - last_ts).dt.days).rename('recency_days')
active_days   = ((last_ts - first_ts).dt.days + 1).rename('active_days')
events_per_day= (feat_df.groupby('client_id').size() / active_days).rename('events_per_day')

# Create a new dataframe with the generated features
user_features = (
    event_counts
    .join(unique_cats)
    .join(price_stats)
    .join(recency_days)
    .join(active_days)
    .join(events_per_day)
    .reset_index()
)



Calculate the events per category for each user and enrich the features

In [86]:

cat_event_counts = (
    feat_df[feat_df['category'].isin(top_categories)]
    .groupby(['client_id', 'category', 'event_type'])
    .size()
    .reset_index(name='count')
)

# Create one column per category-event for each user
cat_event_features = (
    cat_event_counts
      .pivot(
         index='client_id',
         columns=['event_type','category'],
         values='count'
      )
      .fillna(0)
)

# Flatten multi-index columns 
cat_event_features.columns = [f'{event}_cat_{int(cat)}' for event, cat in cat_event_features.columns]
cat_event_features = cat_event_features.reset_index()

# Join with existing user_features
user_features = (
    user_features
      .merge(cat_event_features,
             on='client_id',
             how='left')
      .fillna(0)
)


print("user_features shape:", user_features.shape)
print(user_features.head())

user_features shape: (5610, 58)
   client_id  count_add_to_cart  count_buy  count_page_visit  \
0       9854                  2          1                 0   
1      10827                  3          6                 0   
2      12713                  0          1                 0   
3      16107                  1          4                 0   
4      34316                  9          3                 0   

   count_remove_from_cart  count_search_query  unique_categories  avg_price  \
0                       1                   0                  1  69.000000   
1                       0                   0                  2  66.000000   
2                       0                   0                  1  70.000000   
3                       0                   0                  2  47.750000   
4                       1                   0                  4  33.333333   

   max_price  min_price  ...  buy_cat_5323  remove_from_cart_cat_5323  \
0       69.0       69.0  ...       

In [36]:
# Initialize empty DataFrame to hold per-category ratios
category_behavior_features = pd.DataFrame({'client_id': user_features['client_id']})

# Loop through top categories
for cat in top_categories:
    # Build safe column names
    add_col = f'add_to_cart_cat_{cat}'
    buy_col = f'buy_cat_{cat}'
    rem_col = f'remove_from_cart_cat_{cat}'
    view_col = f'page_visit_cat_{cat}'  # This might not exist for all categories
    
    # Create safe columns (0 if missing)
    for col in [add_col, buy_col, rem_col, view_col]:
        if col not in user_features.columns:
            user_features[col] = 0

    # Compute ratios
    category_behavior_features[f'cart_to_buy_ratio_cat_{cat}'] = user_features[buy_col] / (user_features[add_col] + 1)
    category_behavior_features[f'remove_to_add_ratio_cat_{cat}'] = user_features[rem_col] / (user_features[add_col] + 1)
    category_behavior_features[f'pageviews_per_buy_cat_{cat}'] = user_features[view_col] / (user_features[buy_col] + 1)
    
user_features = user_features.merge(category_behavior_features, on='client_id', how='left')


C:\Users\dimit\AppData\Local\Temp\ipykernel_4836\4157332575.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  user_features[col] = 0
C:\Users\dimit\AppData\Local\Temp\ipykernel_4836\4157332575.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  user_features[col] = 0
C:\Users\dimit\AppData\Local\Temp\ipykernel_4836\4157332575.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.co

# Monetary features

In [38]:
monetary_cat = (
    feat_df[feat_df['event_type'] == 'buy']
    .groupby(['client_id', 'category'])['price']
    .agg(['sum', 'mean', 'count'])
    .reset_index()
    .rename(columns={'sum': 'buy_total', 'mean': 'buy_avg_price', 'count': 'buy_count'})
)

# Keep only top categories
monetary_cat = monetary_cat[monetary_cat['category'].isin(top_categories)]

# Pivot to wide format
monetary_cat_pivot = monetary_cat.pivot(index='client_id', columns='category')
monetary_cat_pivot.columns = [f"{stat}_cat_{int(cat)}" for stat, cat in monetary_cat_pivot.columns]
monetary_cat_pivot = monetary_cat_pivot.reset_index()


In [40]:
# Total buys per user
total_buys_per_user = (
    feat_df[feat_df['event_type'] == 'buy']
    .groupby('client_id')
    .size()
    .rename('total_buys')
)

# Buys per user-category
cat_buy_counts = (
    feat_df[(feat_df['event_type'] == 'buy') & (feat_df['category'].isin(top_categories))]
    .groupby(['client_id', 'category'])
    .size()
    .rename('cat_buy_count')
    .reset_index()
)

# Merge total buys
cat_buy_counts = cat_buy_counts.merge(total_buys_per_user, on='client_id')
cat_buy_counts['buy_ratio'] = cat_buy_counts['cat_buy_count'] / (cat_buy_counts['total_buys'] + 1)

# Pivot to wide
buy_ratio_pivot = cat_buy_counts.pivot(index='client_id', columns='category', values='buy_ratio')
buy_ratio_pivot.columns = [f'buy_ratio_cat_{int(c)}' for c in buy_ratio_pivot.columns]
buy_ratio_pivot = buy_ratio_pivot.reset_index()


# Temporal features per category 

In [42]:
# Add day-of-week and hour
feat_df['hour'] = feat_df['timestamp'].dt.hour
feat_df['dow'] = feat_df['timestamp'].dt.dayofweek
feat_df['is_weekend'] = feat_df['dow'].isin([5, 6]).astype(int)

# Filter to top categories only
temporal_df = feat_df[feat_df['category'].isin(top_categories)]

# Weekend ratio per user & category
weekend_ratio_cat = (
    temporal_df
    .groupby(['client_id', 'category'])['is_weekend']
    .mean()
    .unstack(fill_value=0)
    .rename(columns=lambda c: f'weekend_activity_ratio_cat_{int(c)}')
    .reset_index()
)

# Peak hour per category (optional)
peak_hour_cat = (
    temporal_df
    .groupby(['client_id', 'category'])['hour']
    .agg(lambda x: x.value_counts().idxmax() if len(x) > 0 else 0)
    .unstack(fill_value=0)
    .rename(columns=lambda c: f'peak_hour_cat_{int(c)}')
    .reset_index()
)


# Category Diversity Features 

In [43]:
# Distinct event types per category
event_type_diversity = (
    temporal_df
    .groupby(['client_id', 'category'])['event_type']
    .nunique()
    .unstack(fill_value=0)
    .rename(columns=lambda c: f'event_type_diversity_cat_{int(c)}')
    .reset_index()
)

# Sessions per category (simplified as unique timestamps for now)
session_count_cat = (
    temporal_df
    .groupby(['client_id', 'category'])['timestamp']
    .nunique()
    .unstack(fill_value=0)
    .rename(columns=lambda c: f'session_count_cat_{int(c)}')
    .reset_index()
)


# Merging with the user_features dataframe the new features

In [46]:
user_features = (
    user_features
    .merge(monetary_cat_pivot, on='client_id', how='left')
    .merge(buy_ratio_pivot, on='client_id', how='left')
    .merge(weekend_ratio_cat, on='client_id', how='left')
    .merge(peak_hour_cat, on='client_id', how='left')  # optional, can skip if noisy
    .merge(event_type_diversity, on='client_id', how='left')
    .merge(session_count_cat, on='client_id', how='left')
    .fillna(0)
)


In [48]:
# Compute recency per event_type and category
recency_df = (
    feat_df[feat_df['category'].isin(top_categories)]
    .groupby(['client_id', 'event_type', 'category'])['timestamp']
    .max()
    .reset_index()
)

# Calculate days since the last event (relative to cutoff date)
recency_df['days_since'] = (cutoff_dt - recency_df['timestamp']).dt.days

# Pivot to wide format: one column per event_type-category pair
recency_pivot = recency_df.pivot(index='client_id', columns=['event_type', 'category'], values='days_since')

# Flatten the multi-index column names
recency_pivot.columns = [f'days_since_{evt}_cat_{int(cat)}' for evt, cat in recency_pivot.columns]
recency_pivot = recency_pivot.reset_index()

# Merge with existing user_features
user_features = user_features.merge(recency_pivot, on='client_id', how='left')

# Fill missing values (i.e. user did not perform that event for that category)
user_features.fillna(0, inplace=True)


In [50]:
# Fill the missing values with 0

user_features[['avg_price','max_price','min_price']] = user_features[
    ['avg_price','max_price','min_price']
].fillna(0)



print("user_features shape:", user_features.shape)
print(user_features.head())

user_features shape: (7354, 538)
   client_id  count_add_to_cart  count_buy  count_page_visit  \
0       9854                  2          1                 0   
1      10827                  3          6                 0   
2      12713                  0          1                 0   
3      16107                  1          4                 0   
4      24093                  0          1                 0   

   count_remove_from_cart  count_search_query  unique_categories  avg_price  \
0                       1                   0                  1      69.00   
1                       0                   0                  2      66.00   
2                       0                   0                  1      70.00   
3                       0                   0                  2      47.75   
4                       0                   0                  1      85.00   

   max_price  min_price  ...  days_since_add_to_cart_cat_4981  \
0       69.0       69.0  ...              

Merge the dataframes and check dimensions

In [52]:

label_list = (
    label_buys[ label_buys['category'].isin(top_categories) ]
      .groupby('client_id')['category']
      .apply(list)
      .reset_index(name='label_categories')
)

total_df = user_features.merge(
   label_list,
    on='client_id',
    how='left'
)


print("After merge, total_df shape:", total_df.shape)

After merge, total_df shape: (7354, 539)


Use LightGBM and xgBoost classifiers to predict on the test set 

In [54]:
from sklearn.preprocessing import MultiLabelBinarizer
X = total_df.drop(columns = ['label_categories'])

# Turn list of categories into a binary indicator matrix
mlb = MultiLabelBinarizer(classes=sorted(top_categories)) 
y = mlb.fit_transform(total_df['label_categories'])

In [ ]:
#pip install iterative-stratification

In [58]:
# Multi-label stratified train/test split


from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

msss = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=0.25, random_state=42
)
train_idx, test_idx = next(msss.split(X, y))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx],    y[test_idx]

print("Train/test users:", X_train.shape[0], X_test.shape[0])
print("Train label shape:", y_train.shape, "Test label shape:", y_test.shape)

Train/test users: 5526 1828
Train label shape: (5526, 25) Test label shape: (1828, 25)


In [57]:
'''
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle = True, 
    stratify= None   
)

print("X Train shape:", X_train.shape, "X Test shape:", X_test.shape)
print("y train shape: ", y_train.shape, "y test shape:", y_test.shape)'''

'\nX_train, X_test, y_train, y_test = train_test_split(\n    X, y,\n    test_size=0.2,\n    random_state=42,\n    shuffle = True, \n    stratify= None   \n)\n\nprint("X Train shape:", X_train.shape, "X Test shape:", X_test.shape)\nprint("y train shape: ", y_train.shape, "y test shape:", y_test.shape)'

In [60]:
models = {
    'LightGBM': OneVsRestClassifier(
        LGBMClassifier(
            n_estimators=200,
            learning_rate=0.05,
            num_leaves=31,
            n_jobs=-1,
            random_state=42
        )
    ),
    'XGBoost': OneVsRestClassifier(
        XGBClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=6,
            eval_metric='logloss',
            n_jobs=-1,
            random_state=42
        )
    )
}

In [72]:
#Predict and evaluate

for name, clf in models.items():
    print(f"\n=== {name} ===")
    clf.fit(X_train, y_train)
    
   
    y_prob = clf.predict_proba(X_test)
  
    y_pred = (y_prob > 0.3).astype(int)

   
    auroc = roc_auc_score(y_test, y_prob, average='macro')
    prec  = precision_score(y_test, y_pred, average='macro', zero_division=0)
    rec   = recall_score(y_test, y_pred, average='macro', zero_division=0)
    f1    = f1_score(y_test, y_pred, average='macro', zero_division=0)
    
    ham   = hamming_loss(y_test, y_pred)


    print(f"AUROC (macro):    {auroc:.4f}")
    print(f"Precision (macro):{prec:.4f}")
    print(f"Recall (macro):   {rec:.4f}")
    print(f"F1 (macro):       {f1:.4f}")
    print(f"Hamming loss:     {ham:.4f}")




=== LightGBM ===
[LightGBM] [Info] Number of positive: 229, number of negative: 5297
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003873 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8778
[LightGBM] [Info] Number of data points in the train set: 5526, number of used features: 361
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.041440 -> initscore=-3.141174
[LightGBM] [Info] Start training from score -3.141174
[LightGBM] [Info] Number of positive: 182, number of negative: 5344
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003563 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8778
[LightGBM] [Info] Number of data points in the train set: 5526, number of used features: 361
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.032935 -> initscore=-3.379723
[LightGBM] [Info] Start training from score -3.379723


RANDOM FOREST CLASSIFIER

In [73]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier


rf_clf = OneVsRestClassifier(
    RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    )
)

rf_clf.fit(X_train, y_train)


y_prob_rf = rf_clf.predict_proba(X_test)

# Threshold
threshold = 0.3
y_pred_rf = (y_prob_rf > threshold).astype(int)


print("Random Forest performance (threshold p>0.3):")
print("  AUROC (macro):   ", roc_auc_score(y_test, y_prob_rf, average='macro'))
print("  Precision (mac): ", precision_score(y_test, y_pred_rf, average='macro', zero_division=0))
print("  Recall (macro):  ", recall_score(y_test, y_pred_rf, average='macro', zero_division=0))
print("  F1 (macro):      ", f1_score(y_test, y_pred_rf, average='macro', zero_division=0))
print("  Hamming loss:    ", hamming_loss(y_test, y_pred_rf))

Random Forest performance (threshold p>0.3):
  AUROC (macro):    0.6772388544165124
  Precision (mac):  0.3675632136692614
  Recall (macro):   0.1993400219852197
  F1 (macro):       0.24576305382918817
  Hamming loss:     0.04936542669584245


In [65]:
#Tune hyperparameters
from sklearn.model_selection import RandomizedSearchCV
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

# 5-fold multi label 
cv = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# LightGBM hyperparameter space
lgb_base = OneVsRestClassifier(
    LGBMClassifier(random_state=42),
    n_jobs=-1
)
lgb_param_dist = {
    'estimator__n_estimators':    [100, 200, 500],
    'estimator__learning_rate':   [1e-3, 1e-2, 5e-2, 1e-1],
    'estimator__num_leaves':      [31, 50, 100],
    'estimator__min_child_samples': [5, 10, 20],
    'estimator__subsample':       [0.6, 0.8, 1.0],
    'estimator__colsample_bytree': [0.6, 0.8, 1.0],
}




In [67]:
# Randomized search for LightGBM
lgb_search = RandomizedSearchCV(
    lgb_base,
    lgb_param_dist,
    n_iter=20,
    scoring='roc_auc',
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=2
)
lgb_search.fit(X_train, y_train)

print("Best LightGBM params:", lgb_search.best_params_)
best_lgb = lgb_search.best_estimator_

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best LightGBM params: {'estimator__subsample': 0.8, 'estimator__num_leaves': 31, 'estimator__n_estimators': 500, 'estimator__min_child_samples': 20, 'estimator__learning_rate': 0.01, 'estimator__colsample_bytree': 0.8}


In [69]:
# Tune hyperparameters of xgboost classifier 
xgb_base = OneVsRestClassifier(
    XGBClassifier(
        use_label_encoder=False,
        eval_metric='logloss',
        random_state=42
    ),
    n_jobs=-1
)

# Specify the hyperparameter search space
xgb_param_dist = {
    'estimator__n_estimators':    [100, 200, 500],
    'estimator__learning_rate':   [1e-3, 1e-2, 5e-2, 1e-1],
    'estimator__max_depth':       [3, 6, 10],
    'estimator__subsample':       [0.6, 0.8, 1.0],
    'estimator__colsample_bytree': [0.6, 0.8, 1.0],
}

In [71]:
# Search for the best hyperparameters
from sklearn.model_selection import RandomizedSearchCV
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

cv = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_search = RandomizedSearchCV(
    xgb_base,
    xgb_param_dist,
    n_iter=20,
    scoring='roc_auc',
    cv=cv,                # MultilabelStratifiedKFold instance
    random_state=42,
    n_jobs=-1,
    verbose=2
)

# train
xgb_search.fit(X_train, y_train)

# Inspect best parameters
print("Best XGBoost params:", xgb_search.best_params_)
best_xgb = xgb_search.best_estimator_

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Best XGBoost params: {'estimator__subsample': 1.0, 'estimator__n_estimators': 100, 'estimator__max_depth': 3, 'estimator__learning_rate': 0.05, 'estimator__colsample_bytree': 0.8}


# Evaluating the tuned models 

In [73]:
# Final evaluation using the best parameters on test set
def evaluate(clf, X_test, y_test, threshold=0.2):
    y_prob = clf.predict_proba(X_test)
    y_pred = (y_prob > threshold).astype(int)
    print(f"\n=== {clf.__class__.__name__} @ thresh={threshold} ===")
    print("AUROC (macro):   ", roc_auc_score(y_test, y_prob, average='macro'))
    print("Precision (mac): ", precision_score(y_test, y_pred, average='macro', zero_division=0))
    print("Recall (macro):  ", recall_score(y_test, y_pred, average='macro', zero_division=0))
    print("F1 (macro):      ", f1_score(y_test, y_pred, average='macro', zero_division=0))
    print("Hamming loss:    ", hamming_loss(y_test, y_pred))

evaluate(best_lgb, X_test, y_test)
evaluate(best_xgb, X_test, y_test)


=== OneVsRestClassifier @ thresh=0.2 ===
AUROC (macro):    0.7392240983118498
Precision (mac):  0.40839714269904664
Recall (macro):   0.32309941232525763
F1 (macro):       0.34574514116024313
Hamming loss:     0.08764733616218764

=== OneVsRestClassifier @ thresh=0.2 ===
AUROC (macro):    0.740769817561305
Precision (mac):  0.4618584702584326
Recall (macro):   0.3132668491620914
F1 (macro):       0.3538535524588555
Hamming loss:     0.08104667609618105


ALTERNATIVE APPROACH : PREDICT THE TOP 5 CATEGORIES PER USER 

In [75]:
#tuned hyperparameter model prediction

def evaluate_topk_model(model, model_name, X_test, y_test, mlb, top_k=5):


    # === Predict probabilities ===
    y_prob = model.predict_proba(X_test)

    # === Get true labels and class names ===
    true_label_sets = [set(labels) for labels in mlb.inverse_transform(y_test)]
    category_labels = mlb.classes_

    # === Get top-k predicted classes for each user ===
    top_k_indices = np.argsort(y_prob, axis=1)[:, -top_k:][:, ::-1]
    top_k_pred_sets = [
        set(category_labels[i] for i in row)
        for row in top_k_indices
    ]

    # === Evaluate Recall@k and Precision@k ===
    recall_at_k = [
        len(true & pred) / len(true) if true else 0.0
        for true, pred in zip(true_label_sets, top_k_pred_sets)
    ]
    precision_at_k = [
        len(true & pred) / top_k
        for true, pred in zip(true_label_sets, top_k_pred_sets)
    ]

    # === Output results ===
    avg_recall_at_k = np.mean(recall_at_k)
    avg_precision_at_k = np.mean(precision_at_k)
    macro_auroc = roc_auc_score(y_test, y_prob, average='macro')

    print(f"\n=== Top-{top_k} Evaluation for {model_name} ===")
    print(f"Macro AUROC:        {macro_auroc:.4f}")
    print(f"Average Recall@{top_k}:   {avg_recall_at_k:.4f}")
    print(f"Average Precision@{top_k}: {avg_precision_at_k:.4f}")

    # === Optional: return full DataFrame ===
    return pd.DataFrame({
        'client_id': X_test['client_id'].values,
        'true_categories': [list(labels) for labels in true_label_sets],
        f'top_{top_k}_predicted': [list(pred) for pred in top_k_pred_sets],
        'recall_at_k': recall_at_k,
        'precision_at_k': precision_at_k
    })


In [77]:
results_lgb = evaluate_topk_model(best_lgb, "LightGBM", X_test, y_test, mlb, top_k=5)
results_xgb = evaluate_topk_model(best_xgb, "XGBoost", X_test, y_test, mlb, top_k=5)



=== Top-5 Evaluation for LightGBM ===
Macro AUROC:        0.7392
Average Recall@5:   0.6817
Average Precision@5: 0.1491

=== Top-5 Evaluation for XGBoost ===
Macro AUROC:        0.7408
Average Recall@5:   0.7046
Average Precision@5: 0.1536


In [81]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Add, LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import roc_auc_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
import numpy as np

# === Scale inputs ===
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# === Split train/val ===
X_tr, X_val, y_tr, y_val = train_test_split(X_train_s, y_train, test_size=0.2, random_state=42)

# === Model architecture ===
n_features = X_tr.shape[1]
n_labels   = y_tr.shape[1]

inputs = Input(shape=(n_features,))
x = Dense(512)(inputs)
x = BatchNormalization()(x)
x = LeakyReLU()(x)
x = Dropout(0.4)(x)

x1 = Dense(256)(x)
x1 = BatchNormalization()(x1)
x1 = LeakyReLU()(x1)
x1 = Dropout(0.3)(x1)

x2 = Dense(256)(x1)
x2 = BatchNormalization()(x2)
x2 = LeakyReLU()(x2)
x2 = Dropout(0.3)(x2)

x = Add()([x1, x2])  # residual

x = Dense(128)(x)
x = BatchNormalization()(x)
x = LeakyReLU()(x)
x = Dropout(0.2)(x)

outputs = Dense(n_labels, activation='sigmoid')(x)

model = Model(inputs, outputs)
model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss=BinaryCrossentropy(label_smoothing=0.05),
    metrics=['AUC']
)

# === Callbacks ===
early_stop = EarlyStopping(monitor='val_AUC', mode='max', patience=10, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_AUC', mode='max', factor=0.5, patience=5, min_lr=1e-5)

# === Training loop to track Recall@5 ===
epochs = 50
batch_size = 128
top_k = 5

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    
    model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=1,
        batch_size=batch_size,
        callbacks=[early_stop, reduce_lr],
        verbose=2
    )

    # Top-k evaluation on validation set
    y_val_prob = model.predict(X_val, verbose=0)
    top_k_indices = np.argsort(y_val_prob, axis=1)[:, -top_k:][:, ::-1]
    y_val_pred_topk = np.zeros_like(y_val_prob)
    for i, indices in enumerate(top_k_indices):
        y_val_pred_topk[i, indices] = 1

    recall_at5 = recall_score(y_val, y_val_pred_topk, average='macro', zero_division=0)
    print(f"Recall@5 (val): {recall_at5:.4f}")



Epoch 1/50
27/27 - 5s - 192ms/step - AUC: 0.6190 - loss: 0.5993 - val_AUC: 0.7333 - val_loss: 0.4657 - learning_rate: 0.0010
Recall@5 (val): 0.5471

Epoch 2/50
27/27 - 0s - 16ms/step - AUC: 0.6776 - loss: 0.3802 - val_AUC: 0.7485 - val_loss: 0.3367 - learning_rate: 0.0010
Recall@5 (val): 0.5792

Epoch 3/50
27/27 - 1s - 19ms/step - AUC: 0.7205 - loss: 0.3092 - val_AUC: 0.7645 - val_loss: 0.3084 - learning_rate: 0.0010
Recall@5 (val): 0.5839

Epoch 4/50
27/27 - 0s - 15ms/step - AUC: 0.7491 - loss: 0.2904 - val_AUC: 0.7732 - val_loss: 0.2917 - learning_rate: 0.0010
Recall@5 (val): 0.5936

Epoch 5/50
27/27 - 1s - 20ms/step - AUC: 0.7602 - loss: 0.2853 - val_AUC: 0.7799 - val_loss: 0.2858 - learning_rate: 0.0010
Recall@5 (val): 0.5879

Epoch 6/50
27/27 - 1s - 20ms/step - AUC: 0.7724 - loss: 0.2821 - val_AUC: 0.7853 - val_loss: 0.2805 - learning_rate: 0.0010
Recall@5 (val): 0.5949

Epoch 7/50
27/27 - 1s - 20ms/step - AUC: 0.7782 - loss: 0.2798 - val_AUC: 0.7858 - val_loss: 0.2772 - learning

In [85]:
# Predict on test set
y_prob_test = model.predict(X_test_s, verbose=0)
top_k_indices = np.argsort(y_prob_test, axis=1)[:, -top_k:][:, ::-1]

# Binary prediction matrix
y_pred_top5 = np.zeros_like(y_prob_test)
for i, indices in enumerate(top_k_indices):
    y_pred_top5[i, indices] = 1

# Metrics
macro_auroc = roc_auc_score(y_test, y_prob_test, average='macro')
macro_recall = recall_score(y_test, y_pred_top5, average='macro', zero_division=0)
macro_precision = precision_score(y_test, y_pred_top5, average='macro', zero_division=0)

print("\n=== Final Evaluation on Test Set using the NN ===")
print(f"Macro AUROC:        {macro_auroc:.4f}")
print(f"Macro Recall@5:     {macro_recall:.4f}")
print(f"Macro Precision@5:  {macro_precision:.4f}")



=== Final Evaluation on Test Set using the NN ===
Macro AUROC:        0.7352
Macro Recall@5:     0.5879
Macro Precision@5:  0.1329
